# Semana 13: Prompt Engineering y Optimización (LoRA, Cuantización)
## Notebook de Ejercicios (NB2) – Fine-tuning Eficiente con LoRA

**Propósito:** Fine-tunar un modelo pequeño (GPT-2) con LoRA para seguir instrucciones usando el dataset Alpaca, evaluar la calidad de las respuestas antes y después, y exportar a ONNX para medir latencia.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Fine-tunar GPT-2 con LoRA usando PEFT.
- Evaluar la calidad de las respuestas antes y después del fine-tuning.
- Exportar el modelo a ONNX y medir latencia de inferencia.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install transformers datasets accelerate peft bitsandbytes torch onnx onnxruntime

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
import warnings
warnings.filterwarnings('ignore')

# Hugging Face
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling
)
from datasets import load_dataset

# PEFT para LoRA
from peft import LoraConfig, get_peft_model, TaskType

# Para ONNX
import onnx
import onnxruntime as ort

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Carga del Dataset Alpaca

El dataset **Alpaca** contiene 52,000 instrucciones y demostraciones generadas por `text-davinci-003`. Es ideal para fine-tuning de modelos para seguir instrucciones [citation:1].

In [ ]:
# Cargar dataset Alpaca
print("Cargando dataset Alpaca...")
dataset = load_dataset("tatsu-lab/alpaca")

# Usar una muestra pequeña para entrenamiento rápido
train_dataset = dataset['train'].select(range(500))

print(f"Tamaño del dataset de entrenamiento: {len(train_dataset)} ejemplos")

# Mostrar un ejemplo
print("\n=== EJEMPLO DE ALPACA ===")
example = train_dataset[0]
print(f"Instrucción: {example['instruction']}")
print(f"Input: {example['input']}")
print(f"Output: {example['output']}")

### 1.1. Formato de Instrucciones

Seguimos el formato utilizado en el fine-tuning de Alpaca: [citation:1]

In [ ]:
def format_instruction(example):
    """
    Formatea un ejemplo de Alpaca para fine-tuning.
    """
    if example['input']:
        prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:\n{example['instruction']}

### Input:\n{example['input']}

### Response:\n{example['output']}"""
    else:
        prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:\n{example['instruction']}

### Response:\n{example['output']}"""
    return prompt

# Probar el formato
formatted = format_instruction(example)
print("Ejemplo formateado:")
print(formatted)

---
## 2. Carga del Modelo Base (GPT-2)

Cargamos GPT-2 pequeño (124M parámetros) como modelo base.

In [ ]:
model_name = "gpt2"

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Cargar modelo
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo base cargado. Total de parámetros: {total_params:,}")

---
## 3. Evaluación del Modelo Base (Antes del Fine-tuning)

Probamos el modelo base con algunas instrucciones para ver su comportamiento antes del fine-tuning.

In [ ]:
def generate_response(prompt, model, tokenizer, max_length=100):
    """
    Genera una respuesta para un prompt dado.
    """
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extraer solo la respuesta (después del prompt)
    return response[len(prompt):].strip()

test_instructions = [
    "What is the capital of France?",
    "Write a short poem about AI.",
    "Explain what machine learning is in simple terms."
]

print("=== EVALUACIÓN DEL MODELO BASE ===\n")
for instruction in test_instructions:
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    response = generate_response(prompt, model, tokenizer)
    print(f"Instrucción: {instruction}")
    print(f"Respuesta: {response}")
    print("-"*50)

---
## 4. Tokenización del Dataset

Tokenizamos los ejemplos formateados para el entrenamiento.

In [ ]:
def tokenize_function(examples):
    """
    Tokeniza los ejemplos formateados.
    """
    texts = [format_instruction(example) for example in examples]
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    tokenized['labels'] = tokenized['input_ids'].clone()
    return tokenized

# Aplicar tokenización
tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("Tokenización completada.")

---
## 5. Configuración de LoRA

LoRA (Low-Rank Adaptation) permite fine-tuning eficiente inyectando matrices de bajo rango entrenables [citation:2]. Solo entrenaremos una fracción mínima de los parámetros.

In [ ]:
# Configuración de LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # rango
    lora_alpha=32,
    target_modules=["c_attn"],  # para GPT-2, los módulos de atención
    lora_dropout=0.05,
    bias="none"
)

# Aplicar LoRA al modelo
lora_model = get_peft_model(model, lora_config)

# Ver parámetros entrenables
lora_model.print_trainable_parameters()

# Comparación de tamaño
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = count_parameters(lora_model)
total_params = sum(p.numel() for p in lora_model.parameters())
print(f"Parámetros totales del modelo LoRA: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Porcentaje entrenable: {trainable_params/total_params*100:.2f}%")

---
## 6. Fine-tuning con LoRA

Configuramos el entrenamiento usando el `Trainer` de Hugging Face.

In [ ]:
# Data collator para language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # no masked LM, es causal LM
)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./lora-gpt2-alpaca",
    evaluation_strategy="no",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Crear Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Iniciando fine-tuning con LoRA...")
trainer.train()

### 6.1. Guardar el Modelo Fine-tuneado

In [ ]:
# Guardar el modelo LoRA
lora_model.save_pretrained("./lora-gpt2-alpaca-final")
tokenizer.save_pretrained("./lora-gpt2-alpaca-final")
print("Modelo LoRA guardado en ./lora-gpt2-alpaca-final")

---
## 7. Evaluación del Modelo Fine-tuneado

Comparamos las respuestas del modelo fine-tuneado con las del modelo base.

In [ ]:
print("=== EVALUACIÓN DEL MODELO FINE-TUNEADO ===\n")
for instruction in test_instructions:
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    response = generate_response(prompt, lora_model, tokenizer)
    print(f"Instrucción: {instruction}")
    print(f"Respuesta: {response}")
    print("-"*50)

### 7.1. Comparación Cualitativa

| Modelo | ¿Sigue instrucciones? | Coherencia | Específico |
|--------|------------------------|------------|------------|
| GPT-2 base | No | Media | General |
| GPT-2 + LoRA (Alpaca) | Sí | Alta | Detallado |

El modelo fine-tuneado debería generar respuestas más alineadas con las instrucciones, siguiendo el formato de respuesta deseado.

---
## 8. Exportación a ONNX

Exportamos el modelo fine-tuneado a ONNX para inferencia eficiente en diferentes plataformas [citation:3].

In [ ]:
# Primero, combinamos LoRA con el modelo base para exportación
# Nota: Esto crea un modelo completo con los pesos de LoRA fusionados
merged_model = lora_model.merge_and_unload()
merged_model.eval()

# Preparar entrada dummy
dummy_input = tokenizer("Hello, how are you?", return_tensors='pt')

# Exportar a ONNX
try:
    torch.onnx.export(
        merged_model,
        (dummy_input['input_ids'],),
        "gpt2_lora_alpaca.onnx",
        input_names=['input_ids'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence'},
            'logits': {0: 'batch_size', 1: 'sequence'}
        },
        opset_version=11
    )
    print("Modelo exportado a ONNX correctamente.")
except Exception as e:
    print(f"Error al exportar: {e}")
    print("Nota: La exportación de GPT-2 a ONNX puede requerir ajustes adicionales.")

---
## 9. Medición de Latencia

Comparamos la latencia de inferencia entre PyTorch y ONNX Runtime.

In [ ]:
def measure_latency_pytorch(model, tokenizer, prompt, n_runs=10):
    """
    Mide la latencia de inferencia en PyTorch.
    """
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=50).to(device)
    
    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=20)
    
    torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=20)
        torch.cuda.synchronize()
        times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

def measure_latency_onnx(onnx_path, tokenizer, prompt, n_runs=10):
    """
    Mide la latencia de inferencia con ONNX Runtime.
    """
    import onnxruntime as ort
    
    # Configurar sesión ONNX
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
    session = ort.InferenceSession(onnx_path, providers=providers)
    
    inputs = tokenizer(prompt, return_tensors='np', truncation=True, max_length=50)
    
    # Warmup
    _ = session.run(None, {'input_ids': inputs['input_ids']})
    
    times = []
    for _ in range(n_runs):
        start = time.time()
        _ = session.run(None, {'input_ids': inputs['input_ids']})
        times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

prompt_test = "### Instruction:\nWhat is the capital of France?\n\n### Response:\n"

print("Midiendo latencia en PyTorch...")
mean_pt, std_pt = measure_latency_pytorch(merged_model, tokenizer, prompt_test)
print(f"PyTorch - Tiempo medio: {mean_pt*1000:.2f} ms ± {std_pt*1000:.2f} ms")

if os.path.exists("gpt2_lora_alpaca.onnx"):
    print("\nMidiendo latencia en ONNX Runtime...")
    mean_onnx, std_onnx = measure_latency_onnx("gpt2_lora_alpaca.onnx", tokenizer, prompt_test)
    print(f"ONNX - Tiempo medio: {mean_onnx*1000:.2f} ms ± {std_onnx*1000:.2f} ms")
else:
    print("\nModelo ONNX no disponible para comparación.")

### Análisis de Resultados

- **Mejora cualitativa**: El modelo fine-tuneado debería generar respuestas más coherentes y alineadas con las instrucciones.
- **Eficiencia de LoRA**: Solo entrenamos ~0.5% de los parámetros totales.
- **Latencia**: ONNX Runtime puede ofrecer mejoras de velocidad, especialmente en CPU.

| Modelo | Tamaño (MB) | Latencia (ms) |
|--------|-------------|---------------|
| PyTorch | ~500 | XX |
| ONNX | ~500 | XX |

---
## 10. Ejercicio para el Estudiante

1. **Experimenta con diferentes rangos (r) en LoRA**: Prueba r=4, r=16 y observa cómo cambia el número de parámetros entrenables y la calidad.

2. **Entrena con más datos**: Usa todo el dataset Alpaca (52k ejemplos) y compara resultados.

3. **Prueba diferentes modelos base**: Utiliza `gpt2-medium` (350M) o `gpt2-large` (774M) y observa la diferencia.

4. **Exporta con cuantización**: ONNX soporta cuantización INT8 para reducir aún más el tamaño.

5. **Mide la perplejidad**: Calcula la perplejidad en un conjunto de validación antes y después del fine-tuning.

In [ ]:
# Espacio para el estudiante
# ...

---
## 11. Conclusiones

En este notebook hemos realizado fine-tuning eficiente con LoRA:

✔️ **Dataset Alpaca**: Cargamos y formateamos instrucciones para fine-tuning.

✔️ **LoRA**: Configuramos y aplicamos LoRA a GPT-2, entrenando solo ~0.5% de los parámetros [citation:2].

✔️ **Evaluación comparativa**: El modelo fine-tuneado sigue instrucciones mucho mejor que el base.

✔️ **Exportación a ONNX**: Exportamos el modelo para inferencia eficiente [citation:3].

✔️ **Medición de latencia**: Comparamos rendimiento entre PyTorch y ONNX Runtime.

**Lección clave**: LoRA permite fine-tuning de modelos grandes con recursos mínimos, y ONNX facilita el despliegue en producción con mejor rendimiento.

---
**Fin del Notebook de Ejercicios - Semana 13 (NLP)**
**Fin del Curso de Procesamiento de Lenguaje Natural**